In [1]:
import zipfile
from multiprocessing import cpu_count
import splitfolders
from tensorflow import keras
import numpy as np
from tqdm import tqdm
import shutil
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import regularizers
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model


In [2]:
print("TensorFlow version:", tf.__version__)
print("CPU cores:", tf.config.threading.get_intra_op_parallelism_threads())
print(f"CPU cores detected: {cpu_count()}")

TensorFlow version: 2.20.0
CPU cores: 0
CPU cores detected: 12


In [2]:
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

2.20.0
[]


In [5]:
zip_path = "bovine_breeds.zip"


with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

In [7]:
data_dir = 'BovineBreeds_Augmented'
for cls in os.listdir(data_dir):
    path = os.path.join(data_dir, cls)
    print(f"{cls}: {len(os.listdir(path))} images")

.ipynb_checkpoints: 0 images
Alambadi: 246 images
Amritmahal: 251 images
Ayrshire: 234 images
Banni: 235 images
Bargur: 249 images
Bhadawari: 257 images
Brown_Swiss: 225 images
Dangi: 262 images
Deoni: 246 images
Gir: 244 images
Holstein_Friesian: 244 images
Jaffrabadi: 240 images
Jersey: 253 images
Kangayam: 254 images
Kasargod: 250 images
Kenkatha: 288 images
Kherigarh: 306 images
Krishna_Valley: 186 images
Mehsana: 249 images
Murrah: 223 images
Nagori: 255 images
Nili_Ravi: 255 images
Nimari: 261 images
Ongole: 240 images
Red_Sindhi: 246 images
Sahiwal: 244 images
Surti: 272 images
Tharparkar: 267 images
Umblachery: 269 images


In [8]:
input_folder = "BovineBreeds_Augmented"
output_folder = "split_data"

splitfolders.ratio(input_folder,
                   output=output_folder,
                   seed=42,
                   ratio=(0.7, 0.2, 0.1))

Copying files: 7250 files [00:39, 183.44 files/s]


In [13]:
def remove_class(path):
    class_to_remove = '.ipynb_checkpoints'
    folder_path = os.path.join(path, class_to_remove)

    if os.path.exists(folder_path):
        shutil.rmtree(folder_path)
        print(f"{class_to_remove} removed successfully")
    else:
        print("Class not found")

remove_class('split_data/train')
remove_class('split_data/val')
remove_class('split_data/test')

.ipynb_checkpoints removed successfully
.ipynb_checkpoints removed successfully
.ipynb_checkpoints removed successfully


In [3]:
train_dir = 'split_data/train'

class_names_from_dir = sorted(os.listdir(train_dir))
print(class_names_from_dir)

['Alambadi', 'Amritmahal', 'Ayrshire', 'Banni', 'Bargur', 'Bhadawari', 'Brown_Swiss', 'Dangi', 'Deoni', 'Gir', 'Holstein_Friesian', 'Jaffrabadi', 'Jersey', 'Kangayam', 'Kasargod', 'Kenkatha', 'Kherigarh', 'Krishna_Valley', 'Mehsana', 'Murrah', 'Nagori', 'Nili_Ravi', 'Nimari', 'Ongole', 'Red_Sindhi', 'Sahiwal', 'Surti', 'Tharparkar', 'Umblachery']


In [4]:
IMG_SIZE = (300, 300)


train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input,  # ✅ EfficientNet-specific
    rotation_range=15,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
)


val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input,
)

train_generator = train_datagen.flow_from_directory(
    'split_data/train',
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode='categorical'
)


val_generator = val_datagen.flow_from_directory(
    'split_data/val',
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode='categorical',

)



test_generator = val_datagen.flow_from_directory(
    'split_data/test',
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode='categorical'

)

Found 5062 images belonging to 29 classes.
Found 1439 images belonging to 29 classes.
Found 749 images belonging to 29 classes.


In [17]:
base_model = EfficientNetB3(weights='imagenet', include_top=False, input_shape=(300, 300, 3))

base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x1 = Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
x2 = layers.Dropout(0.2)(x1)
x3 = Dense(64, activation = 'relu', kernel_regularizer= regularizers.l2(0.001))(x2)
x4 = layers.Dropout(0.2)(x3)
x5 = Dense(29, activation='softmax')(x4)
model = Model(inputs=base_model.input, outputs=x5)


model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)    │ (None, 300, 300, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ rescaling_6 (Rescaling)       │ (None, 300, 300, 3)       │               0 │ input_layer_3[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ normalization_3               │ (None, 300, 300, 3)       │               7 │ rescaling_6[0][0]          │
│ (Normalization)               │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ rescaling_7 (Rescaling)       │ (None, 300, 300, 3)       │               0 │ normalization_3[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ stem_conv_pad (ZeroPadding2D) │ (None, 301, 301, 3)       │               0 │ rescaling_7[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ stem_conv (Conv2D)            │ (None, 150, 150, 40)      │           1,080 │ stem_conv_pad[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ stem_bn (BatchNormalization)  │ (None, 150, 150, 40)      │             160 │ stem_conv[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ stem_activation (Activation)  │ (None, 150, 150, 40)      │               0 │ stem_bn[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_dwconv                │ (None, 150, 150, 40)      │             360 │ stem_activation[0][0]      │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_bn                    │ (None, 150, 150, 40)      │             160 │ block1a_dwconv[0][0]       │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_activation            │ (None, 150, 150, 40)      │               0 │ block1a_bn[0][0]           │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_se_squeeze            │ (None, 40)                │               0 │ block1a_activation[0][0]   │
│ (GlobalAveragePooling2D)      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_se_reshape (Reshape)  │ (None, 1, 1, 40)          │               0 │ block1a_se_squeeze[0][0]   │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_se_reduce (Conv2D)    │ (None, 1, 1, 10)          │             410 │ block1a_se_reshape[0][0]   │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_se_expand (Conv2D)    │ (None, 1, 1, 40)          │             44

 Total params: 10,990,412 (41.93 MB)

 Trainable params: 206,877 (808.11 KB)

 Non-trainable params: 10,783,535 (41.14 MB)

In [10]:
# Custom callback to monitor CPU during training
import tensorflow.keras.callbacks as callbacks




checkpoint = ModelCheckpoint(
    filepath="/content/drive/MyDrive/Breed Classification/model/best_model.keras",
    monitor="val_accuracy",        # metric to monitor
    mode="max",                    # because we want maximum accuracy
    save_best_only=True,           # saves only when model improves
    verbose=1
)

early_stop = EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)



history = model.fit(train_generator, validation_data=val_generator,
    epochs=20,
    callbacks=[early_stop, checkpoint],     
    verbose = 1

)

Epoch 1/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.2815 - loss: 3.0034
Epoch 1: val_accuracy improved from None to 0.68103, saving model to /content/drive/MyDrive/Breed Classification/model/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Breed Classification/model/best_model.keras
159/159 ━━━━━━━━━━━━━━━━━━━━ 385s 2s/step - accuracy: 0.4482 - loss: 2.3778 - val_accuracy: 0.6810 - val_loss: 1.4357
Epoch 2/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6427 - loss: 1.5648
Epoch 2: val_accuracy improved from 0.68103 to 0.72967, saving model to /content/drive/MyDrive/Breed Classification/model/best_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Breed Classification/model/best_model.keras
159/159 ━━━━━━━━━━━━━━━━━━━━ 375s 2s/step - accuracy: 0.6537 - loss: 1.5199 - val_accuracy: 0.7297 - val_loss: 1.2444
Epoch 3/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6863 - loss: 1.3529
Epoch 3: val_accuracy improved

In [12]:
model_path = "best_model.keras"

loaded_model = load_model(model_path)

loss, accuracy = loaded_model.evaluate(test_generator)
print(loss)
print(accuracy)

ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 29), output.shape=(None, 30)

In [16]:
num_classes = val_generator.num_classes

print(num_classes)

29


In [14]:
print(loaded_model.output_shape)

(None, 30)
